In [34]:
import mysql.connector

#Modify the connection to include the database name
database_name="Code_with_tim_test_database"
db=mysql.connector.connect(
    host="localhost",
    user="root",
    password="gpt67rSy",
    database=database_name, # <-- We are now connecting to this database
    buffered=True,# best practice - for parallel multiple queries,prevents error,standard practice
)
if db.is_connected():
    print(f"Successfully connected to the '{db.database}' database.")

#We can now create cursor to work within this database
my_cursor=db.cursor()

# You can now execute queries that affect this database,
# like creating tables, inserting data, etc.

Successfully connected to the 'code_with_tim_test_database' database.


In [35]:
my_cursor.execute("""
CREATE TABLE test(
    name VARCHAR(50) NOT NULL,
    created DATETIME NOT NULL,
    gender ENUM("male","female","other") NOT NULL,
    id INT AUTO_INCREMENT NOT NULL,
    PRIMARY KEY(id)
)
""")

db.commit()

#### Topic 2: Inserting Data with DATETIME

**Explanation:**
To insert data, we use `INSERT INTO`. A key point here is using `datetime.now()` to generate the current timestamp for the `created` column. We use `%s` as a placeholder for values, which is the standard for the `mysql.connector` library.

In [36]:
from datetime import datetime

# Insert a new entry
insert_query="INSERT INTO test (name, created, gender) values (%s,%s,%s)"
data=("Kristal",datetime.now(),"male")

my_cursor.execute(insert_query,data)
db.commit()# Important! Saves the changes to the database

print(my_cursor.rowcount,"record inserted.")

1 record inserted.


### Insert more data

In [37]:
data2=("Biraj",datetime.now(),"male")
data3=("Biraj",datetime.now(),"female")

my_cursor.execute(insert_query,data2)
my_cursor.execute(insert_query,data3)

db.commit()
print(my_cursor.rowcount,"record_inserted")

1 record_inserted


In [38]:
my_cursor.execute("SELECT * FROM test")

for row in my_cursor:
    print(row)

('Kristal', datetime.datetime(2026, 3, 23, 11, 13, 1), 'male', 1)
('Biraj', datetime.datetime(2026, 3, 23, 11, 13, 1), 'male', 2)
('Biraj', datetime.datetime(2026, 3, 23, 11, 13, 1), 'female', 3)


#### Topic 3: Selecting Data with Conditions (WHERE, ORDER BY)

**Explanation:**
The `SELECT` statement is used to query data from a table.
- **`WHERE`** : Adds a condition to filter the results.
- **`ORDER BY`** : Sorts the results by a specified column. `ASC` is ascending (default), and `DESC` is descending.
- You can select specific columns by listing them instead of using `*` (which selects all columns).

In [39]:
# Select all males
my_cursor.execute("SELECT * FROM test WHERE gender='male'")

for row in my_cursor:
    print(row)

('Kristal', datetime.datetime(2026, 3, 23, 11, 13, 1), 'male', 1)
('Biraj', datetime.datetime(2026, 3, 23, 11, 13, 1), 'male', 2)


In [40]:
# Select all males, ordered by ID in descending order (largest ID first)

my_cursor.execute("SELECT * FROM test WHERE gender='male' ORDER BY id DESC")

for row in my_cursor:
    print(row)

('Biraj', datetime.datetime(2026, 3, 23, 11, 13, 1), 'male', 2)
('Kristal', datetime.datetime(2026, 3, 23, 11, 13, 1), 'male', 1)


In [41]:
# Check table structure first
my_cursor.execute("DESCRIBE test")
for row in my_cursor:
    print(row)

# Check available data
my_cursor.execute("SELECT * FROM test")
for row in my_cursor:
    print(row)

# Then run your query
my_cursor.execute("SELECT id, name FROM test WHERE gender = 'female' ORDER BY id DESC")
for row in my_cursor:
    print(f"ID: {row[0]}, Name: {row[1]}")

('name', 'varchar(50)', 'NO', '', None, '')
('created', 'datetime', 'NO', '', None, '')
('gender', "enum('male','female','other')", 'NO', '', None, '')
('id', 'int', 'NO', 'PRI', None, 'auto_increment')
('Kristal', datetime.datetime(2026, 3, 23, 11, 13, 1), 'male', 1)
('Biraj', datetime.datetime(2026, 3, 23, 11, 13, 1), 'male', 2)
('Biraj', datetime.datetime(2026, 3, 23, 11, 13, 1), 'female', 3)
ID: 3, Name: Biraj


#### Topic 4: Altering Tables (ADD, DROP, CHANGE)

**Explanation:**
The `ALTER TABLE` command is used to modify the structure of an existing table. We can add new columns, drop (delete) existing ones, or change the name and data type of a column.

In [42]:
# Add a new column called 'food'
my_cursor.execute("ALTER TABLE test ADD COLUMN food VARCHAR(50) NOT NULL")

# Describe the table to see the new structure
my_cursor.execute("DESCRIBE test")
for x in my_cursor:
    print(x)


('name', 'varchar(50)', 'NO', '', None, '')
('created', 'datetime', 'NO', '', None, '')
('gender', "enum('male','female','other')", 'NO', '', None, '')
('id', 'int', 'NO', 'PRI', None, 'auto_increment')
('food', 'varchar(50)', 'NO', '', None, '')


#### **2. Dropping (Removing) a Column**

In [43]:
# Remove the 'food' column
my_cursor.execute("ALTER TABLE test DROP COLUMN food")

# Describe again to confirm it's gone
my_cursor.execute("DESCRIBE test")
for x in my_cursor:
    print(x)

('name', 'varchar(50)', 'NO', '', None, '')
('created', 'datetime', 'NO', '', None, '')
('gender', "enum('male','female','other')", 'NO', '', None, '')
('id', 'int', 'NO', 'PRI', None, 'auto_increment')


**3. Changing a Column Name and Type (with Caution)**
**Explanation:**
To change a column's name and definition, we use `CHANGE`. You must re-declare the data type.
- **Important:** If you change the data type to one that cannot hold the existing data (e.g., changing `VARCHAR(50)` to `VARCHAR(2)` when a name like "Timothy" exists), you will get an error.

In [44]:
# Rename 'name' to 'first_name' and keep its type as VARCHAR(50)
my_cursor.execute("ALTER TABLE test CHANGE name first_name VARCHAR(50)")

In [45]:
# This will cause an error if any name has more than 2 characters
my_cursor.execute("ALTER TABLE test CHANGE first_name first_name VARCHAR(2)")

DatabaseError: 1265 (01000): Data truncated for column 'first_name' at row 1

In [46]:
# This works if no name is longer than 10 characters
my_cursor.execute("ALTER TABLE test CHANGE first_name first_name VARCHAR(10)")
